# xHuBERT Experiment 8: SOTA Comparison
**De tai**: He thong goi y san pham dua tren phan tich giong noi va cam xuc
**Hoc vien**: Nguyen Tan Nhu | **GVHD**: TS. Bui Thanh Hung (IUH)

**Yeu cau**: `Runtime` -> `Change runtime type` -> **T4 GPU** -> Save

## Methods
- Re-implemented: CNN1D+MFCC, LSTM+MFCC, CNN2D+LogMel
- Literature: Wei et al. 2025, Bhanbhro et al. 2025, Waleed & Shaker 2025, etc.

In [ ]:
# Kiem tra GPU
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_mem / 1e9, 1), "GB")
else:
    print("WARNING: GPU not enabled! Runtime -> Change runtime type -> T4 GPU")

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = "/content/drive/MyDrive/xhubert_results/"
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Save directory: {SAVE_DIR}")

In [ ]:
# Install dependencies (Colab has torch, numpy, sklearn, matplotlib)
!pip install -q transformers==4.51.3 librosa huggingface-hub safetensors tqdm seaborn

# Verify versions
import torch, transformers, librosa
print(f"torch:        {torch.__version__}")
print(f"transformers: {transformers.__version__}")
print(f"librosa:      {librosa.__version__}")

In [ ]:
# Upload .py modules to Colab
# Option 1: Upload manually via Files panel (drag & drop)
# Option 2: Clone from repo
# !git clone https://github.com/nhunet/xhubert-experiments.git
# %cd xhubert-experiments

# Verify required files
import os
required = [
    "config.py", "data.py", "features.py", "protocols.py",
    "stats.py", "utils.py",
    "models/__init__.py", "models/ml_classifiers.py",
    "models/xhubert.py", "models/hubert_vanilla.py",
    "models/fusion.py", "models/dl_1d.py", "models/dl_2d.py",
]
for f in required:
    status = "OK" if os.path.exists(f) else "MISSING"
    print(f"  [{status}]  {f}")

In [ ]:
# Download RAVDESS dataset
import os
RAVDESS_PATH = "./RAVDESS"
if not os.path.exists(RAVDESS_PATH):
    print("Downloading RAVDESS ...")
    !wget -q https://zenodo.org/record/1188976/files/Audio_Speech_Actors_01-24.zip
    !unzip -q Audio_Speech_Actors_01-24.zip -d RAVDESS/
    print("Done!")
else:
    print(f"RAVDESS already exists at {RAVDESS_PATH}")
    !find {RAVDESS_PATH} -name "*.wav" | wc -l

In [ ]:
# Set save directory
import config
config.SAVE_DIR = SAVE_DIR
config.CKPT_DIR = os.path.join(SAVE_DIR, "checkpoints")
os.makedirs(config.CKPT_DIR, exist_ok=True)
print(f"Results -> {config.SAVE_DIR}")
print(f"Checkpoints -> {config.CKPT_DIR}")

### Keep Colab Alive
Paste this into your **browser Console** (F12 -> Console) to prevent idle timeout:
```javascript
function ClickConnect() {
    console.log("Keeping alive...");
    document.querySelector("colab-toolbar-button#connect").click()
}
setInterval(ClickConnect, 60000)
```

## Load Dataset

In [ ]:
from data import RavdessDataset
import config

dataset = RavdessDataset(sr=config.SR_HANDCRAFTED)
dataset.print_summary()

## Quick Test

In [ ]:
from experiments.exp8_sota_comparison import run_exp8

df_quick = run_exp8(dataset=dataset, force=True, quick=True)
print("Quick test passed!" if len(df_quick) > 0 else "FAILED!")

## Full Run

In [ ]:
df_exp8 = run_exp8(dataset=dataset, force=False)

# Show re-implemented results
reimpl = df_exp8[df_exp8["Source"] == "re-implemented"]
print("=== Re-implemented Baselines ===")
print(reimpl.groupby(["Model", "Protocol"])["accuracy"].agg(["mean", "std"]).round(2))

# Show literature results
lit = df_exp8[df_exp8["Source"] == "literature"]
print("\n=== Literature Results ===")
print(lit[["Model", "Protocol", "accuracy"]].to_string())

## Combined Comparison Table

In [ ]:
import pandas as pd, os

# Load xHuBERT results
exp3_csv = os.path.join(config.SAVE_DIR, "results_exp3_xhubert.csv")
if os.path.exists(exp3_csv):
    df3 = pd.read_csv(exp3_csv)
    print("=== Full Comparison ===")
    # Aggregate xHuBERT
    for proto in ["5-fold CV", "LOSGO"]:
        sub = df3[df3["Protocol"] == proto]
        if len(sub) > 0:
            print(f"  xHuBERT-full ({proto}): "
                  f"{sub['accuracy'].mean():.2f} +/- {sub['accuracy'].std():.2f}")

    # DL baselines
    if len(reimpl) > 0:
        for proto in ["5-fold CV", "LOSGO"]:
            sub = reimpl[reimpl["Protocol"] == proto]
            for model in sub["Model"].unique():
                ms = sub[sub["Model"] == model]
                print(f"  {model} ({proto}): "
                      f"{ms['accuracy'].mean():.2f} +/- {ms['accuracy'].std():.2f}")
else:
    print("Run Exp3 first for full comparison!")